# **Predicción del Precio de viviendas**

In [ ]:
#pip install scikit-learn

##

### ++ Carga de funciones

In [ ]:
# Función imputación de outlier
# ------

def imputar_valores_extremos(df, variable, metodo='media'):
    """
    Imputa valores extremos en una variable de un DataFrame utilizando la media o la mediana.

    Parámetros:
    df (DataFrame): El DataFrame que contiene la variable a imputar.
    variable (str): El nombre de la variable que deseas imputar.
    metodo (str): La forma de imputación ('media' o 'mediana'). Por defecto es 'media'.

    Retorna:
    DataFrame: El DataFrame con la variable imputada.
    """
    if metodo not in ['media', 'mediana']:
        raise ValueError("El método debe ser 'media' o 'mediana'")

    # Calcular la media o la mediana
    if metodo == 'media':
        valor_imputacion = df[variable].mean()
    else:
        valor_imputacion = df[variable].median()

    # Identificar valores extremos (usando una regla de 3 veces la desviación estándar)
    limite_inferior = df[variable].mean() - 3 * df[variable].std()
    limite_superior = df[variable].mean() + 3 * df[variable].std()

    # Imputar valores extremos
    df[variable] = np.where(
        (df[variable] < limite_inferior) | (df[variable] > limite_superior),
        valor_imputacion,
        df[variable]
    )

    return df



def eliminar_filas_outliers(df, variable):
    """
    Elimina las filas de un DataFrame donde los valores de una variable específica son considerados outliers,
    si el porcentaje de outliers es mayor al X%.

    Parámetros:
    df (DataFrame): El DataFrame que contiene la variable a evaluar.
    variable (str): El nombre de la variable en la que se evaluarán y eliminarán los outliers.

    Retorna:
    DataFrame: Un nuevo DataFrame sin las filas que contienen outliers en la variable especificada.
    """
    # Calcular los límites para considerar un valor como outlier
    q1 = df[variable].quantile(0.25)
    q3 = df[variable].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    # Identificar las filas que son outliers
    outliers = df[(df[variable] < limite_inferior) | (df[variable] > limite_superior)]

    # Calcular el porcentaje de outliers
    porcentaje_outliers = len(outliers) / len(df) * 100

    # Eliminar filas si el porcentaje de outliers es mayor al 10%
    if porcentaje_outliers > 5:
        df_sin_outliers = df[(df[variable] >= limite_inferior) & (df[variable] <= limite_superior)]
    else:
        df_sin_outliers = df.copy()

    return df_sin_outliers


# Función para calcular el MAPE
def calcular_mape(y_real, y_pred):
    # Asegurarse de que los valores reales no tengan ceros
    if any(y_real == 0):
        raise ValueError("Los valores reales no pueden contener ceros para calcular el MAPE.")
    mape = np.mean(np.abs((y_real - y_pred) / y_real)) * 100
    return mape

### Carga de funciones

In [ ]:
import pandas as pd # Dataframes
import numpy as np # Arreglos y matrices
import scipy.stats as stats
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt # Graficos
import seaborn as sns  # para análisis gráficos
import random # para generar aleatorios
from sklearn.ensemble import IsolationForest # para retiro de outlier de forma multivariada
from sklearn.linear_model import ElasticNet


Información del atributo (en orden):

Los datos del precio de la vivienda se han utilizado en muchos documentos de aprendizaje automático que abordan la regresión

- **CRIM**: tasa de criminalidad per cápita por ciudad
- **ZN**: proporción de tierra residencial dividida en zonas para lotes de más de 25,000 pies cuadrados.
- **INDUS**: proporción de acres de negocios no minoristas por ciudad
- **CHAS**: variable ficticia Charles River (= 1 si el tramo limita con el río; 0 en caso contrario)
- **NOX**: concentración de óxidos nítricos (partes por 10 millones)
- **RM**: número medio de habitaciones por vivienda
- **TAX**:tasa de impuesto a la propiedad de valor total
- **EDAD**: proporción de unidades ocupadas por el propietario construidas antes de 1940
- **DIS**: distancias ponderadas a cinco centros de empleo
- **RAD**: índice de accesibilidad a autopistas radiales
- **IMPUESTO**: tasa impositiva sobre el valor total de la propiedad por 10,000
- **PTRATIO**: relación alumno-maestro por localidad
- **B**: 1000 (Bk - 0.63) ^ 2 donde Bk es la proporción de migrantes por ciudad
- **LSTAT**:% menor estado de la población
- **MEDV**: valor medio de viviendas ocupadas por el propietario en $ 1000


In [ ]:
# Creando conexión con google drive
from google.colab import drive
drive.mount('/gdrive')

### Carga de datos

In [ ]:
# uploaded = files.upload()
vivienda= pd.read_csv('/gdrive/MyDrive/Colab Notebooks/practicando Diploma ADS/S2/vivienda.csv')
vivienda.head()

In [ ]:
# Eliminar una columna
vivienda = vivienda.drop('Unnamed: 0', axis=1)

# **1. Análisis Descriptivo**

In [ ]:
vivienda.shape

In [ ]:
vivienda.info()

In [ ]:
# Trasformamos al tipo de variable correcto
vivienda['CHAS'] = vivienda['CHAS'].astype('object')

In [ ]:
var_numerics = vivienda.select_dtypes(include=['number']).columns.tolist()
var_numerics

In [ ]:
vivienda.describe() # describe es una funcion para tener un reporte estadistico basico rapidamente

In [ ]:
vivienda[var_numerics].hist(figsize=(18, 10), layout=(3,5), bins=30)
plt.suptitle('Distribución de Variables Numéricas')
plt.show()

### Pre procesamiento de datos

In [ ]:
# Antes de poder realizar el modelo de regresión lineal debemos ver si existe asociación y de qué tipo es.
# coeficiente de correlación
vivienda[var_numerics].corr()

In [ ]:
# Gráfico de calor de correlaciones
sns.heatmap(abs(vivienda[var_numerics].corr()), annot=True, fmt='.1f', cmap='Blues')
plt.show()

In [ ]:
# Retirando correlaciones altas
# -----
corr_matrix = vivienda[var_numerics].corr()
corr_pairs = corr_matrix.unstack() # Reestructura la matriz de correlación
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)] # Filtra las correlaciones duplicadas y las correlaciones consigo mismas (diagonal)
sorted_corr_pairs = corr_pairs.sort_values(ascending=False).drop_duplicates() # Ordena las correlaciones de mayor a menor
result_df_corr = pd.DataFrame(sorted_corr_pairs, columns=['Correlation']) # Convierte la serie resultante a un DataFrame
result_df_corr.reset_index(inplace=True)
result_df_corr.columns = ['Var1', 'Var2', 'Correlation']
result_df_corr.head(15)


In [ ]:
# Filtrar correlaciones altas con PRECIO
var_df_corr = result_df_corr[~(((result_df_corr['Var1'] == 'PRICE') | (result_df_corr['Var2'] == 'PRICE')) & (result_df_corr['Correlation'].abs() >= 0.6))]
var_df_corr.head(15)

In [ ]:
# Filtrar las filas donde "PRICE" aparece en "Variable 1" o "Variable 2"
price_df_corr = result_df_corr[(result_df_corr['Var1'] == 'PRICE') | (result_df_corr['Var2'] == 'PRICE')]
price_df_corr

In [ ]:
# creamos un nuevo dataframe
# -------

df = vivienda.copy()

# -------

In [ ]:
# 'RAD', 'NOX' tienen correlaciones altas con otras variables

In [ ]:
df = df.drop(['RAD', 'NOX'], axis=1)

In [ ]:
vivienda.shape, df.shape

In [ ]:
var_numerics.remove('RAD')
var_numerics.remove('NOX')
var_numerics

In [ ]:
# Gráfico de dispersión multiple
# par= sns.pairplot(vivienda)
# df.plot(kind="scatter", x='RAD', y='NOX')


Revisando presencia de valores atípicos

In [ ]:
# Crear un gráfico de cajas para todas las variables
df[var_numerics].plot(kind='box', figsize=(15, 6))

# Mostrar el gráfico
plt.title('Gráfico de Cajas para Múltiples Variables')
plt.show()

In [ ]:
for k, v in df[var_numerics].items():
  q1 = v.quantile(0.25)
  q3 = v.quantile(0.75)
  irq = q3 - q1
  v_col = v[(v <= q1 - 1.5 * irq) | (v >= q3 + 1.5 * irq)]
  perc = np.shape(v_col)[0] * 100.0 / np.shape(df)[0]
  print("Column %s outliers = %.2f%%" % (k, perc))

In [ ]:
# CRIM, RAD, B tienen muchos atipicos, ¿qué tratamiento se les deberia dar?

In [ ]:
# Imputando variables con presencia de valores atípicos
# ----

# Imputar valores extremos en la columna 'variable1' usando la mediana
df['ZN_imput'] = df['ZN']
df = imputar_valores_extremos(df, 'ZN_imput', metodo='media')
# ----
var_numerics.append('ZN_imput')
var_numerics

In [ ]:
for k, v in df[var_numerics].items():
  q1 = v.quantile(0.25)
  q3 = v.quantile(0.75)
  irq = q3 - q1
  v_col = v[(v <= q1 - 1.5 * irq) | (v >= q3 + 1.5 * irq)]
  perc = np.shape(v_col)[0] * 100.0 / np.shape(df)[0]
  print("Column %s outliers = %.2f%%" % (k, perc))

In [ ]:
# Calcular asimetría y curtosis antes y después de la imputación
skew_original = skew(df['ZN'], nan_policy='omit')
kurt_original = kurtosis(df['ZN'], nan_policy='omit')

skew_imput = skew(df['ZN_imput'], nan_policy='omit')
kurt_imput = kurtosis(df['ZN_imput'], nan_policy='omit')

# Crear gráfico de distribución comparativo
plt.figure(figsize=(10, 5))
sns.kdeplot(df['ZN'], label='Original', fill=True, color='blue', alpha=0.5)
sns.kdeplot(df['ZN_imput'], label='Imputada', fill=True, color='red', alpha=0.5)

plt.xlabel('Valor de Lights')
plt.ylabel('Densidad')
plt.title('Comparación de Distribuciones: Original vs Imputada')
plt.legend()
plt.show()

# Mostrar métricas
print(f"Asimetría Original: {skew_original:.4f}, Curtosis Original: {kurt_original:.4f}")
print(f"Asimetría Imputada: {skew_imput:.4f}, Curtosis Imputada: {kurt_imput:.4f}")

Winsorización: Limitar los valores extremos a un percentil determinado (por ejemplo, el percentil 95).

In [ ]:
from scipy.stats.mstats import winsorize

df['ZN_imput_winsorized'] = winsorize(df['ZN'], limits=[0, 0.05])  # Recorta el 5% superior
# ----
var_numerics.append('ZN_imput_winsorized')
var_numerics

In [ ]:
# Lista de variables a analizar
variables = {
    'Original': df['ZN'].dropna(),  # Eliminar NaN antes de calcular métricas
    'Imputado (Mediana)': df['ZN_imput'],
    'Imputado (winsorize)': df['ZN_imput_winsorized']
}

# Crear la figura del gráfico
plt.figure(figsize=(10, 6))

for label, data in variables.items():
    sns.kdeplot(data, label=label, fill=True)

plt.legend()
plt.title("Comparación de Distribuciones: Original vs Imputaciones")
plt.xlabel("Valor de ZN")
plt.ylabel("Densidad")
plt.show()

# Crear tabla con métricas de asimetría y curtosis
metricas = []
for label, data in variables.items():
    metricas.append([label, skew(data, nan_policy='omit'), kurtosis(data, nan_policy='omit')])

df_metricas = pd.DataFrame(metricas, columns=['Variable', 'Asimetría', 'Curtosis'])
print(df_metricas)

In [ ]:
for k, v in df[var_numerics].items():
  q1 = v.quantile(0.25)
  q3 = v.quantile(0.75)
  irq = q3 - q1
  v_col = v[(v <= q1 - 1.5 * irq) | (v >= q3 + 1.5 * irq)]
  perc = np.shape(v_col)[0] * 100.0 / np.shape(df)[0]
  print("Column %s outliers = %.2f%%" % (k, perc))

In [ ]:
# Eliminar filas outliers de la columna 'variable1'
df_reduced = df.copy()

df_reduced = eliminar_filas_outliers(df_reduced, 'CRIM')
# df_reduced = eliminar_filas_outliers(df_reduced, 'ZN')
# df_reduced = eliminar_filas_outliers(df_reduced, 'B')

In [ ]:
df_reduced.head()

In [ ]:
for k, v in df_reduced.items():
  q1 = v.quantile(0.25)
  q3 = v.quantile(0.75)
  irq = q3 - q1
  v_col = v[(v <= q1 - 1.5 * irq) | (v >= q3 + 1.5 * irq)]
  perc = np.shape(v_col)[0] * 100.0 / np.shape(df)[0]
  print("Column %s outliers = %.2f%%" % (k, perc))

In [ ]:
df.shape, df_reduced.shape

In [ ]:
df_reduced.shape[0]/df.shape[0]

In [ ]:
# Identificar los registros eliminados
df_removed = df.merge(df_reduced, how='left', indicator=True).query('_merge == "left_only"').drop(columns=['_merge'])

# Crear el histplot para comparar la distribución de CRIM
plt.figure(figsize=(10, 5))
sns.histplot(df_removed["CRIM"], color="purple", label="Eliminados", kde=True, bins=30, alpha=0.6)
sns.histplot(df_reduced["CRIM"], color="cyan", label="Conservados", kde=True, bins=30, alpha=0.6)

# Etiquetas y título
plt.xlabel("Valor de CRIM")
plt.ylabel("Frecuencia")
plt.title("Distribución de CRIM: Eliminados vs. Conservados")
plt.legend()

# Mostrar gráfico
plt.show()

# Calcular métricas estadísticas
original_skew = skew(df["CRIM"], nan_policy='omit')
reduced_skew = skew(df_reduced["CRIM"], nan_policy='omit')

original_kurt = kurtosis(df["CRIM"], fisher=True, nan_policy='omit')  # Fisher=True da kurtosis ajustada (normal=0)
reduced_kurt = kurtosis(df_reduced["CRIM"], fisher=True, nan_policy='omit')

# Mostrar comparación
comparison = pd.DataFrame({
    "Métrica": ["Asimetría", "Curtosis"],
    "Original": [original_skew, original_kurt],
    "Imputado": [reduced_skew, reduced_kurt]
})

print(comparison)

In [ ]:
# Crear DataFrame comparativo
stats_removed = df_removed["PRICE"].describe()
stats_reduced = df_reduced["PRICE"].describe()
price_comparison = pd.DataFrame({
    "Métrica": ["Media", "Mediana", "Desviación Estándar", "Mínimo", "Máximo"],
    "Eliminados": [stats_removed["mean"], stats_removed["50%"], stats_removed["std"], stats_removed["min"], stats_removed["max"]],
    "Conservados": [stats_reduced["mean"], stats_reduced["50%"], stats_reduced["std"], stats_reduced["min"], stats_reduced["max"]]
})

plt.figure(figsize=(12,5))
# Histograma de comparación
sns.histplot(df_removed["PRICE"], color="red", label="Eliminados", kde=True, bins=30, alpha=0.6)
sns.histplot(df_reduced["PRICE"], color="blue", label="Conservados", kde=True, bins=30, alpha=0.6)

plt.xlabel("Precio de Vivienda")
plt.ylabel("Frecuencia")
plt.title("Distribución de Precios: Eliminados vs. Conservados")
plt.legend()
plt.show()

In [ ]:
# Imputación multiple de datos
# ------------
# Aplicar Isolation Forest
# iso_forest = IsolationForest(contamination=0.1, random_state=42)
# vivienda['scores'] = iso_forest.fit_predict(vivienda)

In [ ]:
# Filtrar los datos para eliminar outliers
# df = vivienda[vivienda['scores'] == 1].drop(columns=['scores'])
# df.head()

In [ ]:
df_reduced.columns

### Feature Enginering

In [ ]:
df_reduced[var_numerics].hist(figsize=(18, 10), layout=(3,5), bins=30)
plt.suptitle('Distribución de Variables Numéricas')
plt.show()

In [ ]:
# Graficar scatter plots de todas las variables numéricas
# sns.pairplot(df_reduced[var_numerics], diag_kind="kde")  # `diag_kind="kde"` usa densidad en la diagonal
# plt.suptitle('Matriz de Dispersión de Variables Numéricas', y=1.02)
# plt.show()

**Transformaciones para normalizar distribuciones:**

Algunas variables están sesgadas, lo que puede afectar la calidad del modelo.

In [ ]:
# Para valores muy sesgado
df_reduced["log_CRIM"] = np.log(df_reduced["CRIM"] + 1e-5)
df_reduced["log_ZN"] = np.log(df_reduced["ZN"] + 1e-5)
df_reduced["log_LSTAT"] = np.log(df_reduced["LSTAT"] + 1e-5)

# Para reducir la dispersión en variables con valores extremos
df_reduced["sqrt_DIS"] = np.sqrt(df_reduced["DIS"])
df_reduced["sqrt_INDUS"] = np.sqrt(df_reduced["INDUS"])

# Para standarizar sesgadas
df_reduced['AGE_boxcox'], lam_age = stats.boxcox(df_reduced['AGE'])
df_reduced['DIS_boxcox'], lam_dis = stats.boxcox(df_reduced['DIS'])
df_reduced['LSTAT_boxcox'], lam_lstat = stats.boxcox(df_reduced['LSTAT'])

var_enginering = [
    "log_CRIM", "log_ZN", "log_LSTAT",  # Transformación logarítmica
    "sqrt_DIS", "sqrt_INDUS",  # Transformación raíz cuadrada
    "AGE_boxcox", "DIS_boxcox", "LSTAT_boxcox"  # Transformación Box-Cox
]
var_enginering

In [ ]:
var_numerics = var_numerics + var_enginering
var_numerics

In [ ]:
df_reduced[var_enginering].hist(figsize=(18, 10), layout=(2,4), bins=30, color='green', edgecolor='black', alpha=0.7)
plt.suptitle('Distribución de Variables construidas')
plt.show()

In [ ]:
vivienda.shape, df_reduced.shape

In [ ]:
var_modeler = var_numerics + ['CHAS']
var_modeler

In [ ]:
# Definimmos nuestra variable model para utilizar en el modelado
# --------
# ------

# df_model = vivienda
df_model = df_reduced

print(df_model.shape)
df_model

# **2. Modelos de regresión**

### 3. Regresión lineal múltiple

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Al dataset o set de entrenamiento le retiramos la variable dependiente o target
X = df_model.drop("PRICE",axis=1) # covariables
Y = df_model['PRICE'] # target

In [ ]:
X.columns, Y.name

In [ ]:
# Creamos nuestro set de datos
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=20)

In [ ]:
# Creamos nuestro modelo general de regresión lineal múltiple
lrm = LinearRegression()

In [ ]:
# Ajustamos nuestro modelo a nuestra base train
lrm_model = lrm.fit(X_train, Y_train)

In [ ]:
# Mostramos los coeficientes del modelo
intercepto_m = lrm_model.intercept_
coeficiente_m = lrm_model.coef_

In [ ]:
print("El núnero de coeficientes es :")
len(lrm_model.coef_)

In [ ]:
# Podemos observar todos los coeficientes asignados con sus nombres
coef=pd.DataFrame(lrm_model.coef_,X.columns)
coef

In [ ]:
# Obtenemos las predicciones
X_pred_m = lrm_model.predict(X_train)
Y_pred_m = lrm_model.predict(X_test)

In [ ]:
# Obtenemos las funciones de coste
print ('Error cuadrático medio:')
print('Train:',mean_squared_error(Y_train, X_pred_m))
print('Test:',mean_squared_error(Y_test, Y_pred_m))

In [ ]:
print ('Raiz Error cuadrático medio:')
print('Train:',np.sqrt(mean_squared_error(Y_train, X_pred_m)))
print('Test:',np.sqrt(mean_squared_error(Y_test, Y_pred_m)))

In [ ]:
print ('Error porcentual absoluto medio:')
print('Train:',calcular_mape(Y_train, X_pred_m))
print('Test:',calcular_mape(Y_test, Y_pred_m))

In [ ]:
print ('R cuadrado:')
print('Train:',r2_score(Y_train, X_pred_m))
print('Test:',r2_score(Y_test, Y_pred_m))

### 4. Regresión Penalizada : Ridge

In [ ]:
from sklearn.preprocessing import scale
from sklearn.model_selection import train_test_split # Partición muestral
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

In [ ]:
# Creamos un vector con los valores de alpha o constante de penalización
alphas = 10**np.linspace(10,-1,100)*0.5
print(alphas.shape)
print(alphas[:10])

In [ ]:
# Normaliza los datos manualmente
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Generamos el ajuste de las regresiones para cada valor de alpha
ridge = Ridge()
coefs = []
# Itera sobre los valores de alpha
for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(X_scaled, Y)
    coefs.append(ridge.coef_)

print(np.shape(coefs))


In [ ]:
# Graficamos los valores de alpha
ax = plt.gca()
ax.plot(alphas, coefs)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')

In [ ]:
# Normaliza los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Le asignamos un valor pequeño a alpha y observamos que pasa con el error en el test
ridge = Ridge(alpha = 0.001)
ridge.fit(X_train_scaled, Y_train)            # Ajustamos el modelo ridge

In [ ]:
# print(pd.Series(ridge.coef_, index = X.columns).sort_values(ascending=False)) # Pintamos los coeficientes
print(pd.Series(ridge.coef_, index = X.columns)) # Pintamos los coeficientes

In [ ]:
# Obtenemos las predicciones
X_pred_rg = ridge.predict(X_train_scaled)
Y_pred_rg = ridge.predict(X_test_scaled)

In [ ]:
# Obtenemos las funciones de coste
print ('Error cuadrático medio:')
print('Train:',mean_squared_error(Y_train, X_pred_rg))
print('Test:',mean_squared_error(Y_test, Y_pred_rg),'\n')

print ('Raiz Error cuadrático medio:')
print('Train:',np.sqrt(mean_squared_error(Y_train, X_pred_rg)))
print('Test:',np.sqrt(mean_squared_error(Y_test, Y_pred_rg)),'\n')

print ('Error porcentual absoluto medio:')
print('Train:',calcular_mape(Y_train, X_pred_rg))
print('Test:',calcular_mape(Y_test, Y_pred_rg))

print ('R cuadrado:')
print('Train:',r2_score(Y_train, X_pred_rg))
print('Test:',r2_score(Y_test, Y_pred_rg))

In [ ]:
# Le asignamos un valor grande a alpha y observamos que pasa con el error en el test
ridge2 = Ridge(alpha = 10)
ridge2.fit(X_train_scaled, Y_train)             # Ajuste del modelo ridge

In [ ]:
print(pd.Series(ridge2.coef_, index = X.columns))

In [ ]:
# Obtenemos las predicciones
X_pred_rg2 = ridge2.predict(X_train_scaled)
Y_pred_rg2 = ridge2.predict(X_test_scaled)

In [ ]:
# Obtenemos las funciones de coste
print ('Error cuadrático medio:')
print('Train:',mean_squared_error(Y_train, X_pred_rg2))
print('Test:',mean_squared_error(Y_test, Y_pred_rg2),'\n')

print ('Raiz Error cuadrático medio:')
print('Train:',np.sqrt(mean_squared_error(Y_train, X_pred_rg2)))
print('Test:',np.sqrt(mean_squared_error(Y_test, Y_pred_rg2)),'\n')

print ('Error porcentual absoluto medio:')
print('Train:',calcular_mape(Y_train, X_pred_rg2))
print('Test:',calcular_mape(Y_test, Y_pred_rg2))

print ('R cuadrado:')
print('Train:',r2_score(Y_train, X_pred_rg2))
print('Test:',r2_score(Y_test, Y_pred_rg2))

In [ ]:
# Podemos hallar el mejor valor de alpha por Cv
# ---------------------------------------------
ridgecv = RidgeCV(alphas = alphas, scoring = 'neg_mean_squared_error')
ridgecv.fit(X_train_scaled, Y_train)
ridgecv.alpha_

In [ ]:
# Probamos la regresión Rige con el mejor alpha
ridge3 = Ridge(alpha = ridgecv.alpha_)
ridge3.fit(X_train_scaled, Y_train)             # Ajuste del modelo ridge

In [ ]:
# Obtenemos las predicciones
X_pred_rg3 = ridge3.predict(X_train_scaled)
Y_pred_rg3 = ridge3.predict(X_test_scaled)

In [ ]:
# Obtenemos las funciones de coste
print ('Error cuadrático medio:')
print('Train:',mean_squared_error(Y_train, X_pred_rg3))
print('Test:',mean_squared_error(Y_test, Y_pred_rg3),'\n')

print ('Raiz Error cuadrático medio:')
print('Train:',np.sqrt(mean_squared_error(Y_train, X_pred_rg3)))
print('Test:',np.sqrt(mean_squared_error(Y_test, Y_pred_rg3)),'\n')

print ('Error porcentual absoluto medio:')
print('Train:',calcular_mape(Y_train, X_pred_rg3))
print('Test:',calcular_mape(Y_test, Y_pred_rg3))

print ('R cuadrado:')
print('Train:',r2_score(Y_train, X_pred_rg3))
print('Test:',r2_score(Y_test, Y_pred_rg3))

In [ ]:
print(pd.Series(ridge3.coef_, index = X.columns))

### 5. Regresión Penalizada : Lasso

In [ ]:
from sklearn.preprocessing import scale
from sklearn.model_selection import train_test_split # Partición muestral
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import mean_squared_error

In [ ]:
lasso = Lasso(max_iter = 10000)
coefs = []

for a in alphas:
    lasso.set_params(alpha=a)
    lasso.fit(scale(X_train_scaled), Y_train)
    coefs.append(lasso.coef_)

print(len(coefs))

In [ ]:
# Graficamos los valores de alpha
ax = plt.gca()
ax.plot(alphas*2, coefs)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')

In [ ]:
# Elegimos el mejor o el valor más óptimo de alpha por Cv
lassocv = LassoCV(alphas = None, cv = 10, max_iter = 100000)
lassocv.fit(X_train_scaled, Y_train)
lassocv.alpha_

In [ ]:
lasso = Lasso(alpha=lassocv.alpha_ ,max_iter = 10000)
lasso.fit(X_train_scaled, Y_train)

In [ ]:
# Obtenemos las predicciones
X_pred_lasso = lasso.predict(X_train_scaled)
Y_pred_lasso = lasso.predict(X_test_scaled)

In [ ]:
# Obtenemos las funciones de coste
print ('Error cuadrático medio:')
print('Train:',mean_squared_error(Y_train, X_pred_lasso))
print('Test:',mean_squared_error(Y_test, Y_pred_lasso),'\n')

print ('Raiz Error cuadrático medio:')
print('Train:',np.sqrt(mean_squared_error(Y_train, X_pred_lasso)))
print('Test:',np.sqrt(mean_squared_error(Y_test, Y_pred_lasso)),'\n')

print ('Error porcentual absoluto medio:')
print('Train:',calcular_mape(Y_train, X_pred_lasso))
print('Test:',calcular_mape(Y_test, Y_pred_lasso))

print ('R cuadrado:')
print('Train:',r2_score(Y_train, X_pred_lasso))
print('Test:',r2_score(Y_test, Y_pred_lasso))

### Analizando los errores en la predicción

In [ ]:
# mejor modelo Random Forest Regressor
X_prediccion = X_pred_m

residuos_rfr = Y_train - X_prediccion

In [ ]:
# Crear el gráfico de dispersión
plt.figure(figsize=(10, 6))
sns.scatterplot( x=X_prediccion, y=Y_train, alpha=0.6)
plt.title('Gráfico de Dispersión: Predicción vs Precios reales')
plt.xlabel('Predicción de prcios')
plt.ylabel('Precios reales')
plt.show()

In [ ]:
# Crear histograma de los residuos
plt.figure(figsize=(8,6))
sns.histplot(residuos_rfr, kde=True)  # kde=True dibuja la curva de densidad
plt.title("Histograma de los residuos")
plt.xlabel("Residuos")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
import scipy.stats as stats

# Crear Q-Q plot
plt.figure(figsize=(8,6))
stats.probplot(residuos_rfr, dist="norm", plot=plt)
plt.title("Q-Q plot de los residuos")
plt.show()

### 5. Guardando los modelos para un uso posterior

In [ ]:
import pickle

In [ ]:
filename = '/gdrive/MyDrive/Colab Notebooks/practicando Diploma ADS/S2/lrm_model.sav'
pickle.dump(lrm_model, open(filename, 'wb'))

### 6. Llamamos al modelo ganador

In [ ]:
# load the model from disk
filename = '/gdrive/MyDrive/Colab Notebooks/practicando Diploma ADS/S2/lrm_model.sav'
loaded_model = pickle.load(open(filename, 'rb'))

# **3. Qué podemos hacer con nuestro modelo?**

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# X_train, X_test , Y_train, Y_test

In [ ]:
var_modeler.remove('PRICE')
var_modeler

In [ ]:
# Normaliza los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[var_modeler])

In [ ]:
y_pred = loaded_model.predict(X_scaled)

In [ ]:
Data_analitycs = X
Data_analitycs['Y'] = Y
Data_analitycs['Y_pred'] = y_pred
# --
Data_analitycs

In [ ]:
Data_analitycs['errores'] = Data_analitycs['Y'] - Data_analitycs['Y_pred']
Data_analitycs

### ¿Las viviendas estan a un precio adecuado?

In [ ]:
# Filtrar viviendas subvaloradas
subvaloradas = Data_analitycs[Data_analitycs['Y'] < Data_analitycs['Y_pred']]
subvaloradas

In [ ]:
# Calcular los errores
subvaloradas['error'] = subvaloradas['Y_pred'] - subvaloradas['Y']

# Visualizar la distribución de los errores
plt.figure(figsize=(10, 6))
sns.histplot(subvaloradas['error'], bins=30, kde=True, color='skyblue')
plt.title('Distribución de Errores en Viviendas Subvaloradas')
plt.xlabel('Error (Precio Predicho - Precio Real)')
plt.ylabel('Frecuencia')
plt.axvline(subvaloradas['error'].mean(), color='red', linestyle='--', label=f'Media del Error: {subvaloradas["error"].mean():.2f}')
plt.legend()
plt.show()

In [ ]:
# Calcular el porcentaje de subvaloración
subvaloradas['porcentaje_subvaloracion'] = (subvaloradas['error'] / subvaloradas['Y']) * 100

# Ordenar de mayor a menor subvaloración
subvaloradas_ordenadas = subvaloradas.sort_values(by='porcentaje_subvaloracion', ascending=False)

# Mostrar las primeras filas
print(subvaloradas_ordenadas[['Y', 'Y_pred', 'error', 'porcentaje_subvaloracion']].head(10))


### ¿Cuanto costará una nueva vivienda?

In [ ]:
# uploaded = files.upload()

vivienda_new= pd.read_csv('/gdrive/MyDrive/Colab Notebooks/practicando Diploma ADS/S2/vivienda_new.csv',sep = ';')
vivienda_new.head()

In [ ]:
# Eliminar una columna
vivienda_new = vivienda_new.drop('Unnamed: 0', axis=1)

# Trasformamos al tipo de variable correcto
vivienda_new['CHAS'] = vivienda_new['CHAS'].astype('object')

# Eliminamos columnas con altas correlaciones
vivienda_new = vivienda_new.drop(['RAD', 'NOX'], axis=1)

# Imputaciones
vivienda_new['ZN_imput'] = vivienda_new['ZN']
vivienda_new = imputar_valores_extremos(vivienda_new, 'ZN_imput', metodo='media')

vivienda_new['ZN_imput_winsorized'] = winsorize(vivienda_new['ZN'], limits=[0, 0.05])

# Para valores muy sesgado
vivienda_new["log_CRIM"] = np.log(vivienda_new["CRIM"] + 1e-5)
vivienda_new["log_ZN"] = np.log(vivienda_new["ZN"] + 1e-5)
vivienda_new["log_LSTAT"] = np.log(vivienda_new["LSTAT"] + 1e-5)

# Para reducir la dispersión en variables con valores extremos
vivienda_new["sqrt_DIS"] = np.sqrt(vivienda_new["DIS"])
vivienda_new["sqrt_INDUS"] = np.sqrt(vivienda_new["INDUS"])

# Para standarizar sesgadas
vivienda_new['AGE_boxcox'], lam_age = stats.boxcox(vivienda_new['AGE'])
vivienda_new['DIS_boxcox'], lam_dis = stats.boxcox(vivienda_new['DIS'])
vivienda_new['LSTAT_boxcox'], lam_lstat = stats.boxcox(vivienda_new['LSTAT'])

In [ ]:
vivienda_new.head()

In [ ]:
# Normaliza los datos
scaler = StandardScaler()
X_scaled_new = scaler.fit_transform(vivienda_new[var_modeler])

In [ ]:
y_pred_new = loaded_model.predict(X_scaled_new)

In [ ]:
vivienda_new['Y_pred'] = y_pred_new

In [ ]:
vivienda_new.head()

In [ ]:
#Gracias!